In [12]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import requests

app = FastAPI()

# USGS API endpoint for San Lorenzo River at Big Trees
USGS_API_URL = "https://waterservices.usgs.gov/nwis/iv/?sites=11160500&parameterCd=00060&format=json"

@app.get("/get_data")
async def get_data():
    """
    Fetch real-time discharge data from USGS API.
    """
    try:
        response = requests.get(USGS_API_URL, timeout=10)  # Add timeout for reliability
        response.raise_for_status()  # Raise HTTPError for bad responses (4xx and 5xx)
        data = response.json()

        # Navigate and extract discharge and timestamp
        discharge = (
            data["value"]["timeSeries"][0]["values"][0]["value"][0]["value"]
        )
        timestamp = (
            data["value"]["timeSeries"][0]["values"][0]["value"][0]["dateTime"]
        )

        return {"discharge": discharge, "timestamp": timestamp}
    except requests.exceptions.RequestException as e:
        return JSONResponse(content={"error": f"Request error: {str(e)}"}, status_code=500)
    except KeyError as e:
        return JSONResponse(content={"error": f"Data parsing error: {str(e)}"}, status_code=500)
    except Exception as e:
        return JSONResponse(content={"error": f"Unexpected error: {str(e)}"}, status_code=500)
